In [ ]:
"""
MASTER EQUATION TEST SUITE — Run All Tests
=============================================
χ = ∭(G · M · E · S · T · K · R · Q · F · C) dx dy dt

Upload to Google Colab → Runtime → Run All → Everything versioned.
Every run saved. Even failures. Full audit trail.

REQUIREMENTS: pip install jax jaxlib (Colab has these pre-installed)

David Lowe (POF 2828) | March 2026
"""

In [ ]:
import sys
import os
import time
import json
import hashlib
import platform
from datetime import datetime, timezone

# Add parent to path for imports
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

print("=" * 70)
print("THEOPHYSICS MASTER EQUATION — MODULAR JAX TEST SUITE v2.0")
print("χ = ∭(G · M · E · S · T · K · R · Q · F · C) dx dy dt")
print("=" * 70)
print(f"Started: {datetime.now(timezone.utc).isoformat()}")
print(f"Python: {platform.python_version()}")
print(f"Platform: {platform.platform()}")

# Install JAX if needed (Colab)
try:
    import jax
    import jax.numpy as jnp
    import numpy as np
    jax.config.update("jax_enable_x64", True)
    print(f"JAX: {jax.__version__} ({jax.default_backend()}, {jax.devices()})")
except ImportError:
    import subprocess
    subprocess.check_call(["pip", "install", "jax[cuda12]", "-q"])
    import jax
    import jax.numpy as jnp
    import numpy as np
    jax.config.update("jax_enable_x64", True)
    print(f"JAX: {jax.__version__} ({jax.default_backend()})")

from version_manager import save_run, get_environment

print()

In [ ]:
print("=" * 70)
print("COMPONENT STATUS")
print("=" * 70)

from components import get_all_components
from master_equation import MasterEquation
from config import SEED_PRIMARY, N_VARS, VAR_SHORT

components = get_all_components()
meq = MasterEquation(components)
print(meq.component_report())
print()

# Quick smoke test
key = jax.random.PRNGKey(SEED_PRIMARY)
q0 = jnp.abs(1.0 + 0.1 * jax.random.normal(key, shape=(N_VARS,)))
chi_val, breakdown = meq.chi_with_breakdown(q0)
print(f"Smoke test: chi(q0) = {chi_val:.6f}")
for k, v in breakdown.items():
    print(f"  {k}: {v:.6f}")
print()

In [ ]:
SUITE_START = datetime.now(timezone.utc)
all_results = {}

# --- Test 1A: Chi-Field Propagation ---
print("\n" + "#" * 70)
print("# PRIORITY 1A: Chi-Field Propagation")
print("#" * 70)
try:
    from tests.test_1a_propagation import run_test_1a
    all_results["test_1a"] = run_test_1a()
except Exception as e:
    print(f"TEST 1A FAILED: {e}")
    import traceback
    traceback.print_exc()
    all_results["test_1a"] = {"verdict": "ERROR", "error": str(e)}

# --- Test 1B: Hubble Gradient ---
print("\n" + "#" * 70)
print("# PRIORITY 1B: Hubble Gradient Prediction")
print("#" * 70)
try:
    from tests.test_1b_hubble import run_test_1b
    all_results["test_1b"] = run_test_1b()
except Exception as e:
    print(f"TEST 1B FAILED: {e}")
    import traceback
    traceback.print_exc()
    all_results["test_1b"] = {"verdict": "ERROR", "error": str(e)}

# --- Test 2A: Veto Property ---
print("\n" + "#" * 70)
print("# PRIORITY 2A: Veto Property Stress Test")
print("#" * 70)
try:
    from tests.test_2a_veto import run_test_2a
    all_results["test_2a"] = run_test_2a()
except Exception as e:
    print(f"TEST 2A FAILED: {e}")
    import traceback
    traceback.print_exc()
    all_results["test_2a"] = {"verdict": "ERROR", "error": str(e)}

# --- Test 2B: Asymmetry ---
print("\n" + "#" * 70)
print("# PRIORITY 2B: Asymmetry Term Effects")
print("#" * 70)
try:
    from tests.test_2b_asymmetry import run_test_2b
    all_results["test_2b"] = run_test_2b()
except Exception as e:
    print(f"TEST 2B FAILED: {e}")
    import traceback
    traceback.print_exc()
    all_results["test_2b"] = {"verdict": "ERROR", "error": str(e)}

# --- Test 3: Coherence Collapse ---
print("\n" + "#" * 70)
print("# PRIORITY 3: Coherence Collapse Model")
print("#" * 70)
try:
    from tests.test_3_coherence_collapse import run_test_3
    all_results["test_3"] = run_test_3()
except Exception as e:
    print(f"TEST 3 FAILED: {e}")
    import traceback
    traceback.print_exc()
    all_results["test_3"] = {"verdict": "ERROR", "error": str(e)}

# --- Test 4: Kolmogorov Sin ---
print("\n" + "#" * 70)
print("# PRIORITY 4: Kolmogorov Sin Validation")
print("#" * 70)
try:
    from tests.test_4_kolmogorov import run_test_4
    all_results["test_4"] = run_test_4()
except Exception as e:
    print(f"TEST 4 FAILED: {e}")
    import traceback
    traceback.print_exc()
    all_results["test_4"] = {"verdict": "ERROR", "error": str(e)}

# --- Test 5: Modified Heisenberg ---
print("\n" + "#" * 70)
print("# PRIORITY 5: Modified Heisenberg")
print("#" * 70)
try:
    from tests.test_5_heisenberg import run_test_5
    all_results["test_5"] = run_test_5()
except Exception as e:
    print(f"TEST 5 FAILED: {e}")
    import traceback
    traceback.print_exc()
    all_results["test_5"] = {"verdict": "ERROR", "error": str(e)}

In [ ]:
print("\n" + "#" * 70)
print("# MODULAR COMPONENT TESTS (each of the 10)")
print("#" * 70)

# Test each component independently
component_results = {}
for key in VAR_SHORT:
    comp = components[key]
    q_test = jnp.array(1.0)

    try:
        val = float(comp.evaluate(q_test, t=0.0))
        # Disable and check it returns 1.0
        comp.enabled = False
        disabled_val = float(comp.evaluate(q_test, t=0.0))
        comp.enabled = True

        ok = disabled_val == 1.0 and val != 1.0
        component_results[key] = {
            "name": comp.name,
            "enabled_value": round(val, 6),
            "disabled_value": round(disabled_val, 6),
            "modular": ok,
        }
        status = "OK" if ok else "ISSUE"
        print(f"  [{status}] {key} ({comp.name}): enabled={val:.4f}, disabled={disabled_val:.4f}")
    except Exception as e:
        component_results[key] = {"error": str(e)}
        print(f"  [ERR] {key}: {e}")

all_modular = all(r.get("modular", False) for r in component_results.values())
print(f"\n  All components modular: {all_modular}")

all_results["component_modularity"] = {
    "verdict": "PASS" if all_modular else "FAIL",
    "per_component": component_results,
}

In [ ]:
SUITE_END = datetime.now(timezone.utc)
duration = (SUITE_END - SUITE_START).total_seconds()

print("\n" + "=" * 70)
print("FINAL SCORECARD")
print("=" * 70)
print(f"{'Test':<40} {'Verdict':<20}")
print("-" * 60)

passes = 0
fails = 0
errors = 0
for name, result in all_results.items():
    v = result.get("verdict", "UNKNOWN")
    print(f"  {name:<38} {v}")
    if "PASS" in str(v).upper():
        passes += 1
    elif "ERROR" in str(v).upper():
        errors += 1
    else:
        fails += 1

print("-" * 60)
print(f"  PASS: {passes}  |  FAIL/PARTIAL: {fails}  |  ERROR: {errors}")
print(f"  Total time: {duration:.1f}s")
print()

# Suite checksum
suite_data = {
    "results": all_results,
    "environment": get_environment(),
    "duration_seconds": duration,
}
suite_str = json.dumps(suite_data, sort_keys=True, default=str)
suite_hash = hashlib.sha256(suite_str.encode()).hexdigest()

print(f"SUITE CHECKSUM: {suite_hash}")
print()

# Save full suite
save_run("full_suite", {"version": "2.0.0-modular"}, suite_data,
         status="pass" if errors == 0 else "partial")

# Save human-readable summary
summary_path = os.path.join(os.path.dirname(os.path.abspath(__file__)),
                            "SUITE_RESULTS.txt")
with open(summary_path, "w") as f:
    f.write("THEOPHYSICS MASTER EQUATION — MODULAR TEST SUITE RESULTS\n")
    f.write(f"chi = triple_integral(G*M*E*S*T*K*R*Q*F*C) dx dy dt\n")
    f.write(f"David Lowe (POF 2828) | {SUITE_START.strftime('%Y-%m-%d')}\n")
    f.write(f"Suite checksum: {suite_hash}\n")
    f.write("=" * 60 + "\n\n")
    for name, result in all_results.items():
        v = result.get("verdict", "UNKNOWN")
        s = result.get("summary", "")
        f.write(f"{name}: {v}\n")
        if s:
            f.write(f"  {s}\n")
        f.write("\n")
    f.write(f"\nTotal: {passes} PASS, {fails} FAIL/PARTIAL, {errors} ERROR\n")
    f.write(f"Duration: {duration:.1f}s\n")
    f.write(f"Checksum: {suite_hash}\n")

print(f"Results saved to: {summary_path}")
print()
print("=" * 70)
print("chi = triple_integral(G * M * E * S * T * K * R * Q * F * C) dx dy dt")
print("Every component modular. Every run versioned. Run it yourself.")
print("=" * 70)